# Phase 6: prepare training-only cross-validation and preprocessing

This phase carries the already locked whole-match holdout forward. It creates ten grouped folds from the training matches only, checks a reusable preprocessing pipeline inside each fold, and saves a small event-to-fold map. It does **not** fit KNN, Naive Bayes, or any other predictive model, and it does not evaluate the held-out test partition.

## 1. Select and check the project environment

Run this notebook with the project's `.venv` kernel in VS Code. The check below works whether Jupyter starts in the project root or in `notebooks/`. It stops early if a different Python environment is active.

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.venv').is_dir() and (p / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not find the project folder containing both .venv/ and README.md.')
sys.path.insert(0, str(project_root))
expected_prefix = (project_root / '.venv').resolve()
active_prefix = Path(sys.prefix).resolve()
if active_prefix != expected_prefix:
    raise RuntimeError(f'Select the project .venv kernel. Expected {expected_prefix}; active Python is {active_prefix}.')
print(f'Project root: {project_root}')
print(f'Python executable: {sys.executable}')
print(f'Virtual environment: {active_prefix}')

## 2. Create or verify the fold preparation

The script first verifies the Phase 4 shot-file fingerprint and the Phase 5 holdout lock. It then reads the source CSV row by row and keeps only rows whose `match_id` belongs to the locked training partition. Match labels are used only for stratified cross-validation within training. Every match is assigned to validation exactly once; all other training matches form that fold's training side. The held-out match IDs are not put into a fold.

The existing CV map is checked against its source, split, contract, and scikit-learn-version lock. If any locked input or map has changed, stop and investigate rather than creating a replacement split.

In [ ]:
import json
import subprocess

script = project_root / 'scripts' / 'prepare_modeling.py'
result = subprocess.run([sys.executable, str(script), '--root', str(project_root)], text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Modeling preparation failed; read the error above before continuing.')

## 3. Inspect fold sizes and class counts

These rows describe the ten validation folds built only from training matches. Confirm that validation shot counts sum to the training shot count and validation match counts sum to the training match count. Each fold must include goals and non-goals on both its training and validation sides. The counts are preparation diagnostics, not model scores.

In [ ]:
import pandas as pd
from IPython.display import display

with (project_root / 'docs' / 'phase_06_preparation_report.json').open(encoding='utf-8') as f:
    preparation_report = json.load(f)
cv_summary = pd.read_csv(project_root / 'reports' / 'tables' / 'phase06_cv_summary.csv')
display(cv_summary)
print(f"Training scope: {preparation_report['training_shot_count']:,} shots in {preparation_report['training_match_count']} matches")
print(f"Validation rows covered: {cv_summary['validation_shot_count'].sum():,}")
print(f"Validation matches covered: {cv_summary['validation_match_count'].sum()}")
assert cv_summary['validation_shot_count'].sum() == preparation_report['training_shot_count']
assert cv_summary['validation_match_count'].sum() == preparation_report['training_match_count']


## 4. Understand the preprocessing plan

There are exactly nine approved inputs: distance and angle are standardized; four text categories are one-hot encoded; three provider flags pass through as 0/1. Each fold gets a fresh transformer fitted only on that fold's training rows. The fitted categories then define the validation encoding; an unseen validation category is safely ignored and its count is recorded. Missing required values cause a clear failure instead of automatic imputation.

The dense one-hot output is usable by Gaussian Naive Bayes, though its Gaussian likelihood is an approximation for indicator columns. Scaling and one-hot dimensions also affect KNN distances, so later model comparisons must explain their preprocessing choices. No classifier has been fitted yet. The Phase 7 model evaluation will reuse this exact CV membership and fit all transformations inside each training fold.

In [ ]:
from scripts.prepare_modeling import make_preprocessor, load_cv_membership

cv_membership = load_cv_membership(project_root)
print(f'Locked event-fold rows: {len(cv_membership):,}')
print('Columns:', list(cv_membership.columns))
print('Fold sizes:')
display(cv_membership.groupby('validation_fold').agg(shots=('event_id', 'size'), matches=('match_id', 'nunique')))
fresh_unfitted_preprocessor = make_preprocessor()
print('A fresh, unfitted ColumnTransformer is available for later fold-local model pipelines.')

## 5. See fold-local scaling in one fold

This small demonstration fits a fresh transformer on fold 1's training rows, then shows a few original and transformed numeric values. The displayed means and scales come from that fold's training rows only, not from all 7,334 training shots. The validation rows are used only for the earlier transform check; no model is trained.

In [ ]:
import json
from scripts.prepare_modeling import FEATURE_COLUMNS, NUMERIC_COLUMNS, load_training_rows, make_preprocessor

with (project_root / 'docs' / 'split_manifest.json').open(encoding='utf-8') as f:
    split_lock = json.load(f)
training_rows = load_training_rows(project_root, set(split_lock['train_match_ids']))
first_fold_validation_ids = set(cv_membership.loc[cv_membership['validation_fold'] == 1, 'event_id'])
first_fold_train = training_rows.loc[~training_rows['event_id'].isin(first_fold_validation_ids)].reset_index(drop=True)
preprocessor = make_preprocessor().fit(first_fold_train[FEATURE_COLUMNS], first_fold_train['is_goal'])
numeric_transformer = preprocessor.named_transformers_['numeric']
print('Fold 1 training-only numeric means:', dict(zip(NUMERIC_COLUMNS, numeric_transformer.mean_)))
print('Fold 1 training-only numeric scales:', dict(zip(NUMERIC_COLUMNS, numeric_transformer.scale_)))
print('First three raw numeric rows:')
display(first_fold_train[NUMERIC_COLUMNS].head(3))
transformed_preview = preprocessor.transform(first_fold_train[FEATURE_COLUMNS].head(3))
print('First three transformed numeric rows:')
display(pd.DataFrame(transformed_preview[:, :2], columns=NUMERIC_COLUMNS))
encoded_names = preprocessor.named_transformers_['categorical'].get_feature_names_out()
print(f'One-hot columns: {len(encoded_names)} total; first names: {list(encoded_names[:10])}')

## Saved artifacts

- `data/processed/cv_membership.csv`: event and match identifiers with one validation-fold number.
- `docs/cv_manifest.json`: source/split/feature-contract fingerprints, library version, algorithm, and fold counts.
- `reports/tables/phase06_cv_summary.csv`: training and validation shot/match/goal counts and validation rates.
- `docs/phase_06_preparation_report.json`: feature roles, preprocessing policy, unseen-category counts, and fold diagnostics.

The script exports no transformed matrices and fits no classifier. Save this notebook after running all cells, then take a screenshot of the CV summary for the project record.